# 10 — Figuras cuantitativas definitivas de InterGATE V8.1

Este notebook genera las cinco figuras principales directamente desde
`results_v8` de la ejecución V8.1. **No contiene tablas ni cifras
incrustadas** y se detiene si detecta resultados V7, una versión distinta
del protocolo o una mezcla entre los resultados de 04, 08b y 09.

No reentrena el modelo ni recalcula pruebas estadísticas. Las 49
permutaciones de etiquetas no son necesarias para estas cinco figuras. Sí
necesita los controles ligeros del notebook 12 (factorial, curva `k/R` y
sensibilidad a subconjuntos de semillas), que se generan ejecutando 12 con
`INTERGATE_RUN_LABEL_PERMUTATIONS=0`.

Cada figura conserva su código visible y los objetos `fig1`–`fig5` para
poder editar títulos, tamaños, colores y anotaciones.


In [ ]:
from pathlib import Path
import json
import math
import os
import sys

os.environ.setdefault('MPLCONFIGDIR', '/tmp/intergate-mpl')

import matplotlib.pyplot as plt
from matplotlib import patheffects
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, Rectangle
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    display = print

    def Markdown(value):
        return value


# Se acepta abrir Jupyter desde la raíz V8.1 o desde notebooks/.
HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in (HERE, *HERE.parents)
        if (path / 'intergate').is_dir()
        and (path / 'configs' / 'protocol_v8.json').is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'No se encuentra una raíz InterGATE V8.1. Abra Jupyter desde '
        'InterGATE_V8_1_matched_budget o desde su carpeta notebooks/.'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.config import CFG
from intergate.runtime_setup import assert_transport_protocol, assert_v8_protocol

assert_v8_protocol(CFG)
assert_transport_protocol(CFG)

EXPECTED_MODEL_VERSION = '8.1.0'
EXPECTED_PROTOCOL = 'intergate-v8-biology-first-v2-matched-budget'
if str(CFG.MODEL_VERSION) != EXPECTED_MODEL_VERSION:
    raise RuntimeError(
        f'Versión incorrecta: {CFG.MODEL_VERSION!r}; '
        f'esperada={EXPECTED_MODEL_VERSION!r}.'
    )
if str(CFG.V8_PROTOCOL_VERSION) != EXPECTED_PROTOCOL:
    raise RuntimeError(
        f'Protocolo incorrecto: {CFG.V8_PROTOCOL_VERSION!r}; '
        f'esperado={EXPECTED_PROTOCOL!r}.'
    )

# Deliberadamente no busca rutas V6/V7 ni resultados alternativos.
RESULTS_ROOT = PROJECT_ROOT / 'results_v8'
OUTPUT_ROOT = RESULTS_ROOT / '10_manuscript_figures'
FIGURES = OUTPUT_ROOT / 'figures'
TABLES = OUTPUT_ROOT / 'tables'
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print('Proyecto:', PROJECT_ROOT)
print('Resultados:', RESULTS_ROOT)
print('Salida de figuras:', FIGURES)
print('InterGATE:', CFG.MODEL_VERSION)
print('Protocolo:', CFG.V8_PROTOCOL_VERSION)


## Estilo global, preflight y carga de resultados V8.1

La celda siguiente permite modificar el estilo común y después valida las
entradas. El preflight no acepta resultados alternativos ni valores
incrustados. Si faltan las tres tablas ligeras de 12, el mensaje indica cómo
generarlas sin iniciar las permutaciones.


In [ ]:
# ===================== EDITAR AQUÍ: ESTILO GLOBAL =====================
STYLE = {
    'font_family': 'DejaVu Sans',
    'base_font_size': 10.0,
    'figure_title_size': 16,
    'panel_title_size': 11.5,
    'axis_label_size': 10.5,
    'tick_label_size': 9.5,
    'legend_size': 8.5,
    'annotation_size': 8.5,
    'gene_label_size': 8.0,
    'dpi': 600,
}

CLASS_COLORS = {
    'LumA': '#4C78A8',
    'LumB': '#72B7B2',
    'HER2-enriched': '#F58518',
    'Basal-like': '#C44E52',
}
NEUTRAL = '#BDBDBD'
DARK_NEUTRAL = '#4D4D4D'
HEATMAP_CMAP = 'magma'
DIAGONAL_COLOR = '#00BFC4'

FIGSIZE = {
    'structural': (12.2, 8.6),
    'basal_core': (14.2, 8.2),
    'biological_validation': (12.4, 8.8),
    'external_concordance': (8.4, 5.0),
    'controls_sensitivity': (15.0, 8.8),
}

plt.rcParams.update({
    'font.family': STYLE['font_family'],
    'font.size': STYLE['base_font_size'],
    'axes.titleweight': 'bold',
    'axes.labelsize': STYLE['axis_label_size'],
    'axes.titlesize': STYLE['panel_title_size'],
    'xtick.labelsize': STYLE['tick_label_size'],
    'ytick.labelsize': STYLE['tick_label_size'],
    'legend.fontsize': STYLE['legend_size'],
    'axes.grid': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

def export_figure(fig, stem):
    """Exporta de forma atómica y mantiene la figura abierta para editarla."""
    for extension in ('pdf', 'svg', 'png'):
        target = FIGURES / f'{stem}.{extension}'
        temporary = FIGURES / f'.{stem}.{extension}.tmp'
        options = {'format': extension, 'bbox_inches': 'tight'}
        if extension == 'png':
            options['dpi'] = STYLE['dpi']
        fig.savefig(temporary, **options)
        os.replace(temporary, target)

def component_sets(edge_pairs):
    """Componentes débiles de una lista de pares dirigidos."""
    nodes = sorted({node for edge in edge_pairs for node in edge})
    parent = {node: node for node in nodes}

    def find(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(left, right):
        root_left, root_right = find(left), find(right)
        if root_left != root_right:
            parent[root_right] = root_left

    for left, right in edge_pairs:
        union(left, right)
    groups = {}
    for node in nodes:
        groups.setdefault(find(node), set()).add(node)
    return sorted(groups.values(), key=lambda values: (-len(values), sorted(values)))

def force_layout(nodes, edges, seed=42):
    """Layout Fruchterman–Reingold pequeño y determinista."""
    rng = np.random.default_rng(seed)
    n_nodes = len(nodes)
    index = {node: position for position, node in enumerate(nodes)}
    positions = rng.normal(0, 0.35, size=(n_nodes, 2))
    optimal_distance = 1.25 / math.sqrt(max(n_nodes, 1))
    temperature = 0.14
    for step in range(320):
        delta = positions[:, None, :] - positions[None, :, :]
        distance = np.sqrt((delta**2).sum(axis=2)) + np.eye(n_nodes)
        repulsion = (
            (delta / distance[:, :, None])
            * ((optimal_distance**2) / np.maximum(distance, 1e-3))[:, :, None]
        )
        force = repulsion.sum(axis=1)
        for source, target, _ in edges:
            source_index, target_index = index[source], index[target]
            vector = positions[source_index] - positions[target_index]
            length = max(float(np.linalg.norm(vector)), 1e-3)
            attraction = (vector / length) * (length**2 / optimal_distance)
            force[source_index] -= attraction
            force[target_index] += attraction
        norms = np.linalg.norm(force, axis=1)
        positions += (
            force / np.maximum(norms[:, None], 1e-9)
            * np.minimum(norms, temperature)[:, None]
        )
        positions -= positions.mean(axis=0)
        temperature = 0.14 * (1 - (step + 1) / 330)
    span = np.ptp(positions, axis=0)
    positions = positions / np.maximum(span, 1e-9)
    return {node: positions[index[node]] for node in nodes}

def clean_term(term):
    return (
        term.replace('HALLMARK_', '')
        .replace('_', ' ')
        .replace('PI3K AKT MTOR', 'PI3K-AKT-mTOR')
        .replace('G2M', 'G2/M')
        .title()
        .replace('Pi3K-Akt-Mtor', 'PI3K-AKT-mTOR')
        .replace('Myc', 'MYC')
        .replace('E2F', 'E2F')
        .replace('G2/M', 'G2/M')
    )

CLASSES = ['LumA', 'LumB', 'HER2-enriched', 'Basal-like']
DISPLAY_CLASSES = ['LumA', 'LumB', 'HER2', 'Basal-like']
EDGE_TYPES = {0: 'HuRI', 1: 'OmniPath activation', 2: 'OmniPath inhibition'}

# ================================================================
# PREFLIGHT EXCLUSIVO DE V8.1
# ================================================================
LOCKED = RESULTS_ROOT / '04_controls' / 'locked_biology_score_w0075'
BIOLOGY = RESULTS_ROOT / '05_biology_validation'
EXTERNAL = RESULTS_ROOT / '08b_external_structural_concordance'
ABLATION = RESULTS_ROOT / '09_ablation_biology_w0075'
REVIEW = RESULTS_ROOT / '12_reviewer_controls'

required = [
    LOCKED / 'locked_score_manifest.json',
    LOCKED / 'biology_panels_locked.json',
    LOCKED / 'biology_consensus' / 'biology_stability_summary.json',
    BIOLOGY / 'locked_signature_manifest.json',
    BIOLOGY / 'matched_candidate_signature_tests.csv',
    BIOLOGY / 'cross_class_specificity.csv',
    BIOLOGY / 'directed_degree_type_sign_rewiring.csv',
    BIOLOGY / 'rewiring_coherence_tests.csv',
    BIOLOGY / 'hallmark_c2_ora_complete_family.csv.gz',
    EXTERNAL / 'external_concordance_matched_null_summary.csv',
    EXTERNAL / 'external_structural_concordance_manifest.json',
    ABLATION / 'ablation_biology_summary.csv',
    ABLATION / 'ablation_matched_signature_tests.csv',
    ABLATION / 'ablation_biology_w0075_manifest.json',
    REVIEW / 'factorial_directional_signature_tests.csv',
    REVIEW / 'factorial_paired_contrasts.csv',
    REVIEW / 'core_threshold_k_over_R.csv',
    REVIEW / 'core_strict_subset_sensitivity.csv',
]
for class_name in CLASSES:
    required.append(
        LOCKED / 'biology_consensus' / f'biology_consensus_{class_name}.csv'
    )
    for seed in (1234, 2025, 7, 99, 31337):
        required.append(
            LOCKED / 'seed_tables' / f'seed_{seed}'
            / f'biology_edges_{class_name}.csv'
        )

missing = [path for path in required if not path.is_file()]
if missing:
    missing_text = '\n- '.join(str(path) for path in missing)
    review_missing = any(REVIEW in path.parents for path in missing)
    hint = ''
    if review_missing:
        hint = (
            '\n\nFaltan controles ligeros de 12. Ejecute antes:\n'
            'INTERGATE_RUN_LABEL_PERMUTATIONS=0 jupyter nbconvert '
            '--execute --to notebook --inplace '
            'notebooks/12_controles_revision_q_permutaciones.ipynb '
            '--ExecutePreprocessor.timeout=-1'
        )
    raise FileNotFoundError('Faltan entradas V8.1:\n- ' + missing_text + hint)


# Verifica los manifiestos que distinguen esta ejecución V8.1 de V7.
EXPECTED_LOCK_SHA256 = (
    '37b37ea295ffadd0b1ef5e3b76aa7a5d2d4d7f53074780eeecb812899e06f077'
)
ablation_manifest = json.loads(
    (ABLATION / 'ablation_biology_w0075_manifest.json').read_text(encoding='utf-8')
)
external_manifest = json.loads(
    (EXTERNAL / 'external_structural_concordance_manifest.json').read_text(
        encoding='utf-8'
    )
)
for name, manifest in (
    ('09', ablation_manifest),
    ('08b', external_manifest),
):
    observed_sha = str(manifest.get('protocol_lock_sha256', ''))
    if observed_sha != EXPECTED_LOCK_SHA256:
        raise RuntimeError(
            f'{name}: protocol_lock_sha256 incompatible: {observed_sha!r}. '
            'No se generarán figuras con resultados mezclados.'
        )
if int(ablation_manifest.get('full_reference_strict_edges', -1)) != 299:
    raise RuntimeError(
        'El manifiesto 09 no corresponde a V8.1 revisada: se esperaban '
        '299 aristas Basal-like 5/5.'
    )
if int(external_manifest.get('n_strict_edges', -1)) != 299:
    raise RuntimeError(
        'El manifiesto 08b no corresponde al núcleo V8.1 de 299 aristas.'
    )


# ================================================================
# CARGA DINÁMICA: NINGÚN RESULTADO ESTÁ EMBEBIDO
# ================================================================
from intergate import final_figures as ff

# Fija explícitamente las rutas del módulo a esta raíz V8.1.
ff.ROOT = PROJECT_ROOT
ff.RESULTS = RESULTS_ROOT
ff.MANUSCRIPT_OUT = OUTPUT_ROOT
ff.TABLES = TABLES
ff.FIGURES = FIGURES
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

summary_structural, strict_edges = ff.structural_tables()
components = ff.basal_component_table(strict_edges['Basal-like'])
matched, cross, rewired = ff.biological_tables()
controls = ff.final_control_tables()

# Compatibilidad de nombres con el código visible de las figuras.
if 'fdr_global' not in matched.columns:
    if 'fdr_global_48_tests' not in matched.columns:
        raise KeyError('No se encuentra la FDR global de los 48 contrastes de 05.')
    matched = matched.rename(columns={'fdr_global_48_tests': 'fdr_global'})

ora_complete = pd.read_csv(BIOLOGY / 'hallmark_c2_ora_complete_family.csv.gz')
hallmark_raw = ora_complete.loc[
    ora_complete['panel'].eq('strict_5of5::Basal-like')
    & ora_complete['collection'].eq('Hallmark')
    & ora_complete['fdr'].lt(0.05)
    & ora_complete['meets_min_overlap'].astype(bool)
].copy()

external_raw = controls['external_structural_concordance']
ablation_raw = controls['basal_single_component_ablation_summary']
ablation_tests = controls['basal_single_component_ablation_tests']
factorial = controls['factorial_q_directional_signature_tests']
threshold = controls['core_threshold_k_over_R']
subsets = controls['core_strict_subset_sensitivity']

# Contrato cruzado 04 ↔ 08b ↔ 09.
basal_strict_edges = len(strict_edges['Basal-like'])
full_strict_edges = int(
    ablation_raw.loc[ablation_raw['arm'].eq('FULL'), 'strict_edges'].iloc[0]
)
if not (basal_strict_edges == full_strict_edges == 299):
    raise RuntimeError(
        'Mezcla de versiones detectada: '
        f'04={basal_strict_edges}, 09={full_strict_edges}, esperado=299.'
    )

source_audit = pd.DataFrame([
    {'check': 'model_version', 'value': str(CFG.MODEL_VERSION)},
    {'check': 'protocol', 'value': str(CFG.V8_PROTOCOL_VERSION)},
    {'check': 'protocol_lock_sha256', 'value': EXPECTED_LOCK_SHA256},
    {'check': 'Basal-like strict 5/5 edges', 'value': basal_strict_edges},
    {'check': 'label permutations required', 'value': False},
])
display(source_audit)
display(summary_structural)


## Figura 1 — Reproducibilidad y topología

Todo el código de los cuatro paneles está en la siguiente celda.

In [ ]:
# ===================== EDITAR AQUÍ: FIGURA 1 =====================
FIG1_TITLE = 'Reproducibility and topology of class-specific networks'
FIG1_PANEL_TITLES = [
    'A  Five-seed reproducibility',
    'B  Connected structure at a common edge budget',
    'C  Largest components of the Basal-like 5/5 core',
    'D  Basal-like core composition',
]
FIG1_BAR_WIDTH = 0.68
FIG1_COMPARISON_WIDTH = 0.36
FIG1_COMPONENT_COLOR = CLASS_COLORS['Basal-like']
FIG1_COMPOSITION_COLORS = [
    '#8C8C8C', '#55A868', '#C44E52', '#4C78A8', '#F58518'
]

fig1, axes = plt.subplots(
    2, 2,
    figsize=FIGSIZE['structural'],
    constrained_layout=True,
)
x = np.arange(len(CLASSES))
class_palette = [CLASS_COLORS[c] for c in CLASSES]
indexed = summary_structural.set_index('class_name').loc[CLASSES]

# Panel A
ax = axes[0, 0]
values = indexed['mean_pairwise_edge_jaccard']
bars = ax.bar(x, values, color=class_palette, width=FIG1_BAR_WIDTH)
for bar, cls in zip(bars, CLASSES):
    n_edges = int(indexed.loc[cls, 'strict_5of5_directed_edges'])
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.015,
        f'{n_edges} edges',
        ha='center',
        fontsize=STYLE['annotation_size'],
    )
ax.set_xticks(x, DISPLAY_CLASSES)
ax.set_ylim(0, max(0.69, float(values.max()) + 0.08))
ax.set_ylabel('Mean pairwise edge Jaccard')
ax.set_title(FIG1_PANEL_TITLES[0], loc='left')
ax.spines[['top', 'right']].set_visible(False)

# Panel B
ax = axes[0, 1]
width = FIG1_COMPARISON_WIDTH
ax.bar(
    x - width/2,
    indexed['consensus_giant_fraction'],
    width,
    color=class_palette,
    label='Exact-K consensus',
)
ax.bar(
    x + width/2,
    indexed['strict_core_giant_fraction'],
    width,
    color=class_palette,
    alpha=0.42,
    hatch='//',
    label='Strict 5/5 edge core',
)
ax.set_xticks(x, DISPLAY_CLASSES)
ax.set_ylabel('Fraction of nodes in largest component')
ax.set_title(FIG1_PANEL_TITLES[1], loc='left')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)

# Panel C
ax = axes[1, 0]
top_components = components.head(10).sort_values(
    'component_rank', ascending=False
)
bars = ax.barh(
    [f'C{i}' for i in top_components['component_rank']],
    top_components['n_genes'],
    color=FIG1_COMPONENT_COLOR,
    alpha=0.88,
)
for bar, n_edges in zip(bars, top_components['n_directed_edges']):
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height()/2,
        f'{int(n_edges)} edges',
        va='center',
        fontsize=STYLE['annotation_size'],
    )
ax.set_xlabel('Genes in Basal-like strict component')
ax.set_title(FIG1_PANEL_TITLES[2], loc='left')
ax.spines[['top', 'right']].set_visible(False)

# Panel D
ax = axes[1, 1]
basal_strict = strict_edges['Basal-like']
edge_counts = (
    basal_strict['edge_type'].map(EDGE_TYPES).value_counts()
    .reindex(EDGE_TYPES.values()).fillna(0)
)
direction = pd.Series({
    'Class-strengthened': int(
        (basal_strict['median_differential_logit'] >= 0).sum()
    ),
    'Class-attenuated': int(
        (basal_strict['median_differential_logit'] < 0).sum()
    ),
})
labels = list(edge_counts.index) + list(direction.index)
counts = list(edge_counts.astype(int)) + list(direction.astype(int))
bars = ax.barh(
    labels[::-1], counts[::-1], color=FIG1_COMPOSITION_COLORS[::-1]
)
for bar in bars:
    ax.text(
        bar.get_width() + 3,
        bar.get_y() + bar.get_height()/2,
        f'{int(bar.get_width())}',
        va='center',
        fontsize=STYLE['annotation_size'],
    )
ax.set_xlim(0, max(counts) * 1.18)
ax.set_xlabel('Strict 5/5 directed edges')
ax.set_title(FIG1_PANEL_TITLES[3], loc='left')
ax.spines[['top', 'right']].set_visible(False)

fig1.suptitle(
    FIG1_TITLE,
    fontsize=STYLE['figure_title_size'],
    fontweight='bold',
)
export_figure(fig1, 'Figure_structural_recovery')
display(fig1)

## Figura 2 — Organización del núcleo Basal-like 5/5

Puedes editar la lista de genes etiquetados, el tamaño de los nodos, los
colores de nodo y arista, la curvatura de las flechas y las leyendas.

In [ ]:
largest_nodes

In [ ]:
# ===================== EDITAR AQUÍ: FIGURA 2 =====================
FIG2_TITLE = 'Organization of the Basal-like 5/5 edge core'
FIG2_LABELLED_GENES = {"VGLL1", "LMO4", "TEAD4", "E2F3", "CYB5R2", "CDC5L"}
FIG2_NODE_COLORS = {
    'signature_and_hallmark': '#C44E52',
    'signature_only': '#F58518',
    'hallmark_only': '#8064A2',
    'other': '#C7C7C7',
}
FIG2_EDGE_COLORS = {
    0: '#9E9E9E',
    1: '#55A868',
    2: '#C44E52',
}
FIG2_NODE_BASE_SIZE = 65
FIG2_NODE_DEGREE_SCALE = 20
FIG2_EDGE_ALPHA = 0.42
FIG2_EDGE_WIDTH = 0.65
FIG2_EDGE_CURVATURE = 0.04

strict = strict_edges['Basal-like']
edge_rows = [
    (row.source_gene, row.target_gene, int(row.edge_type))
    for row in strict.itertuples(index=False)
]
largest_nodes = component_sets(
    [(source, target) for source, target, _ in edge_rows]
)[0]
sub_edges = [
    (source, target, edge_type)
    for source, target, edge_type in edge_rows
    if source in largest_nodes and target in largest_nodes
]
sub_nodes = sorted(largest_nodes)

hallmark = hallmark_raw.copy().sort_values('fold_enrichment', ascending=True)
basal_core_gene_count = len(set(strict['source_gene']) | set(strict['target_gene']))

hallmark_genes = set()
for value in hallmark['overlap_genes'].dropna():
    hallmark_genes.update(str(value).split(';'))
signature_genes = set(
    str(components.iloc[0]['basal_signature_genes']).split(';')
)

fig2 = plt.figure(
    figsize=FIGSIZE['basal_core'],
    constrained_layout=True,
)
grid = fig2.add_gridspec(1, 2, width_ratios=[1.65, 1.0])

# Panel A: red dirigida
ax = fig2.add_subplot(grid[0, 0])
positions = force_layout(sub_nodes, sub_edges)
degree = {node: 0 for node in sub_nodes}
for source, target, _ in sub_edges:
    degree[source] += 1
    degree[target] += 1

node_colors = []
node_sizes = []
for node in sub_nodes:
    in_signature = node in signature_genes
    in_hallmark = node in hallmark_genes
    if in_signature and in_hallmark:
        node_colors.append(FIG2_NODE_COLORS['signature_and_hallmark'])
    elif in_signature:
        node_colors.append(FIG2_NODE_COLORS['signature_only'])
    elif in_hallmark:
        node_colors.append(FIG2_NODE_COLORS['hallmark_only'])
    else:
        node_colors.append(FIG2_NODE_COLORS['other'])
    node_sizes.append(
        FIG2_NODE_BASE_SIZE + FIG2_NODE_DEGREE_SCALE * degree[node]
    )

for edge_type, color in FIG2_EDGE_COLORS.items():
    for source, target, current_type in sub_edges:
        if current_type != edge_type:
            continue
        arrow = FancyArrowPatch(
            positions[source],
            positions[target],
            arrowstyle='-|>',
            mutation_scale=5.5,
            linewidth=FIG2_EDGE_WIDTH,
            color=color,
            alpha=FIG2_EDGE_ALPHA,
            connectionstyle=f'arc3,rad={FIG2_EDGE_CURVATURE}',
            shrinkA=3.2,
            shrinkB=3.2,
        )
        ax.add_patch(arrow)

coordinates = np.vstack([positions[node] for node in sub_nodes])
ax.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=node_colors,
    s=node_sizes,
    linewidths=0.45,
    edgecolors='white',
    zorder=3,
)
for node in sub_nodes:
    if node not in FIG2_LABELLED_GENES:
        continue
    label = ax.text(
        positions[node][0],
        positions[node][1],
        node,
        fontsize=STYLE['gene_label_size'],
        fontweight='bold',
        ha='center',
        va='center',
        zorder=4,
    )
    label.set_path_effects([
        patheffects.withStroke(linewidth=2.5, foreground='white')
    ])

ax.set_title(
    f'A  Largest Basal-like strict component\n'
    f'{len(sub_nodes)} genes, {len(sub_edges)} directed edges',
    loc='left',
)
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='',
           markerfacecolor=FIG2_NODE_COLORS['signature_and_hallmark'],
           markeredgecolor='white', markersize=7,
           label='Basal signature + Hallmark'),
    Line2D([0], [0], marker='o', linestyle='',
           markerfacecolor=FIG2_NODE_COLORS['signature_only'],
           markeredgecolor='white', markersize=7,
           label='Basal signature only'),
    Line2D([0], [0], marker='o', linestyle='',
           markerfacecolor=FIG2_NODE_COLORS['hallmark_only'],
           markeredgecolor='white', markersize=7,
           label='Hallmark only'),
    Line2D([0], [0], marker='o', linestyle='',
           markerfacecolor=FIG2_NODE_COLORS['other'],
           markeredgecolor='white', markersize=7,
           label='Neither'),
    Line2D([0], [0], color=FIG2_EDGE_COLORS[0], lw=1.2, label='HuRI'),
    Line2D([0], [0], color=FIG2_EDGE_COLORS[1], lw=1.2,
           label='OmniPath activation'),
    Line2D([0], [0], color=FIG2_EDGE_COLORS[2], lw=1.2,
           label='OmniPath inhibition'),
]
ax.legend(
    handles=legend_handles,
    loc='lower right',
    frameon=False,
    fontsize=7.3,
    ncol=2,
    handletextpad=0.5,
    columnspacing=0.9,
)
ax.axis('off')

# Panel B: Hallmarks
right = grid[0, 1].subgridspec(2, 1, height_ratios=[1.18, 1.0])
ax2 = fig2.add_subplot(right[0, 0])
bars = ax2.barh(
    [clean_term(term) for term in hallmark['term']],
    hallmark['fold_enrichment'],
    color=FIG2_NODE_COLORS['hallmark_only'],
)
for bar, overlap, fdr in zip(
    bars, hallmark['overlap_n'], hallmark['fdr']
):
    ax2.text(
        bar.get_width() + 0.08,
        bar.get_y() + bar.get_height()/2,
        f'n={int(overlap)}; FDR={fdr:.3g}',
        va='center',
        fontsize=STYLE['annotation_size'],
    )
ax2.set_xlim(0, float(hallmark['fold_enrichment'].max()) * 1.42)
ax2.set_xlabel('Fold enrichment')
ax2.set_title(
    f'B  Significant Hallmarks in the {basal_core_gene_count}-gene 5/5 core',
    loc='left',
)
ax2.spines[['top', 'right']].set_visible(False)

# Panel C: perfil de componentes
ax3 = fig2.add_subplot(right[1, 0])
top_components = components.head(10).sort_values(
    'component_rank', ascending=False
)
bars = ax3.barh(
    [f'C{i}' for i in top_components['component_rank']],
    top_components['n_genes'],
    color=CLASS_COLORS['Basal-like'],
)
for bar, n_signature in zip(
    bars, top_components['basal_signature_genes_n']
):
    ax3.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height()/2,
        f'{int(n_signature)} signature genes',
        va='center',
        fontsize=STYLE['annotation_size'],
    )
ax3.set_xlim(0, float(top_components['n_genes'].max()) * 1.35)
ax3.set_xlabel('Genes')
ax3.set_title('C  Component-size profile', loc='left')
ax3.spines[['top', 'right']].set_visible(False)

fig2.suptitle(
    FIG2_TITLE,
    fontsize=STYLE['figure_title_size'],
    fontweight='bold',
)
export_figure(fig2, 'Figure_Basal_strict_core')
display(fig2)


In [ ]:
# ============================================================================
# FIGURA SUPLEMENTARIA: NÚCLEO BASAL-LIKE 5/5 CONECTADO POR EL PRIOR COMPLETO
# ============================================================================
#
# Añadir como una nueva celda al notebook 10 después de ejecutar las celdas de
# preflight y carga. No modifica el núcleo 5/5 ni el modelo: crea únicamente una
# expansión para visualización usando caminos reales del prior HuRI-OmniPath.
#
# Estrategia:
#   1. Calcula las 75 componentes débiles del núcleo estricto.
#   2. Busca caminos mínimos en el prior completo (distancia débil/no dirigida).
#   3. En empates de saltos, prioriza el producto de confianza más alto.
#   4. Construye un bosque mínimo entre componentes (cierre métrico + Kruskal).
#   5. Mantiene separadas las componentes sin ningún camino en el prior.
#
# Las 299 aristas estrictas se muestran continuas. Las aristas añadidas desde
# el prior se muestran discontinuas y nunca se contabilizan como núcleo 5/5.

from __future__ import annotations

import heapq
import json
import math
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.patheffects as patheffects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch


# ============================= EDITAR AQUÍ =============================
SUPP_CONNECT_TITLE = (
    "Prior-supported completion of the Basal-like strict 5/5 edge core"
)
SUPP_CONNECT_LAYOUT_SEED = 42
SUPP_CONNECT_FIGSIZE = (15.5, 12.5)

# Ruta habitual dentro de V8.1. Solo cambiar si se movió el resultado de 05.
SUPP_CONNECT_PRIOR_PATH = BIOLOGY / "candidate_prior_for_null.csv.gz"

SUPP_CONNECT_LABEL_ALL_CORE_GENES = False
SUPP_CONNECT_LABEL_CONNECTOR_GENES = True
SUPP_CONNECT_LABEL_UNREACHABLE_GENES = True

SUPP_CONNECT_LABELLED_GENES = {
    "VGLL1",
    "LMO4",
    "TEAD4",
    "E2F3",
    "CYB5R2",
    "CDC5L",
    "PLK1",
    "AURKB",
    "EGFR",
}

SUPP_CONNECT_NODE_COLORS = {
    "signature_and_hallmark": "#C44E52",
    "signature_only": "#F58518",
    "hallmark_only": "#8064A2",
    "other": "#C7C7C7",
    "connector": "#4C78A8",
}
SUPP_CONNECT_CORE_EDGE_COLORS = {
    0: "#929292",  # HuRI
    1: "#3B8F5A",  # OmniPath activation
    2: "#C43C39",  # OmniPath inhibition
}
SUPP_CONNECT_BRIDGE_EDGE_COLOR = "#2B6CB0"
SUPP_CONNECT_UNREACHABLE_OUTLINE = "#111111"

SUPP_CONNECT_NODE_BASE_SIZE = 22
SUPP_CONNECT_NODE_DEGREE_SCALE = 5.3
SUPP_CONNECT_CORE_EDGE_WIDTH = 0.46
SUPP_CONNECT_CORE_EDGE_ALPHA = 0.45
SUPP_CONNECT_BRIDGE_EDGE_WIDTH = 0.88
SUPP_CONNECT_BRIDGE_EDGE_ALPHA = 0.82
SUPP_CONNECT_SELECTED_LABEL_SIZE = 7.7
SUPP_CONNECT_AUX_LABEL_SIZE = 6.4
SUPP_CONNECT_ALL_LABEL_SIZE = 2.8

# Ajustes manuales solo para etiquetas próximas entre sí, en puntos.
SUPP_CONNECT_LABEL_OFFSETS = {
    "JAK1": (-28, 15),
    "SRC": (-28, -14),
    "VGLL1": (-22, 10),
    "TEAD4": (18, -7),
    "LMO4": (19, -25),
}

# Salidas tabulares reproducibles.
SUPP_CONNECT_TABLE_DIR = TABLES / "basal_prior_completion"
# ======================================================================


def weak_components(edge_pairs):
    """Componentes débiles deterministas de pares de genes."""

    nodes = sorted({node for edge in edge_pairs for node in edge})
    parent = {node: node for node in nodes}

    def find(node):
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(left, right):
        root_left, root_right = find(left), find(right)
        if root_left != root_right:
            parent[root_right] = root_left

    for left, right in edge_pairs:
        union(left, right)

    groups = defaultdict(set)
    for node in nodes:
        groups[find(node)].add(node)
    return sorted(
        groups.values(),
        key=lambda values: (-len(values), tuple(sorted(values))),
    )


def normalize_prior(prior):
    """Colapsa pares para calcular distancias débiles y conserva metadatos."""

    required = {
        "source_gene",
        "target_gene",
        "edge_type",
        "prior_weight",
        "candidate_edge_id",
    }
    missing = required - set(prior.columns)
    if missing:
        raise KeyError(f"Faltan columnas en el prior: {sorted(missing)}")

    pair_meta = {}
    for row in prior.itertuples(index=False):
        source = str(row.source_gene)
        target = str(row.target_gene)
        if source == target:
            continue
        edge_type = int(row.edge_type)
        prior_weight = float(row.prior_weight)
        candidate_edge_id = int(row.candidate_edge_id)
        sign = int(getattr(row, "sign", 1 if prior_weight >= 0 else -1))
        pair_key = tuple(sorted((source, target)))

        # Confianza para el camino: magnitud del peso; la dirección se mantiene
        # en los metadatos y en la flecha final, pero no bloquea conectividad.
        confidence = min(max(abs(prior_weight), 1e-12), 1.0)
        rank = (
            confidence,
            int(edge_type in (1, 2)),
            -candidate_edge_id,
            source,
            target,
        )
        candidate = {
            "source_gene": source,
            "target_gene": target,
            "edge_type": edge_type,
            "prior_weight": prior_weight,
            "sign": sign,
            "candidate_edge_id": candidate_edge_id,
            "confidence": confidence,
            "penalty": -math.log(confidence),
            "rank": rank,
        }
        if pair_key not in pair_meta or rank > pair_meta[pair_key]["rank"]:
            pair_meta[pair_key] = candidate

    adjacency = defaultdict(list)
    for pair_key, metadata in pair_meta.items():
        left, right = pair_key
        penalty = metadata["penalty"]
        adjacency[left].append((right, penalty, pair_key))
        adjacency[right].append((left, penalty, pair_key))
    for node in adjacency:
        adjacency[node].sort(key=lambda value: (value[1], value[0]))
    return pair_meta, adjacency


def lexicographic_component_paths(core_components, adjacency):
    """Caminos: primero mínimo número de saltos, después máxima confianza."""

    component_of = {
        node: component_index
        for component_index, component in enumerate(core_components)
        for node in component
    }
    best_pair_paths = {}

    for source_component, source_nodes in enumerate(core_components):
        best_cost = {}
        previous = {}
        queue = []
        for node in sorted(source_nodes):
            best_cost[node] = (0, 0.0)
            previous[node] = None
            heapq.heappush(queue, (0, 0.0, node))

        reached_components = {source_component}
        while queue:
            hops, penalty, node = heapq.heappop(queue)
            if (hops, penalty) != best_cost.get(node):
                continue

            target_component = component_of.get(node)
            if (
                target_component is not None
                and target_component not in reached_components
            ):
                reached_components.add(target_component)
                path_nodes = [node]
                cursor = node
                while previous[cursor] is not None:
                    cursor = previous[cursor]
                    path_nodes.append(cursor)
                path_nodes.reverse()

                left, right = sorted((source_component, target_component))
                canonical_path = min(
                    tuple(path_nodes),
                    tuple(reversed(path_nodes)),
                )
                candidate = {
                    "component_a": left,
                    "component_b": right,
                    "hops": hops,
                    "penalty": penalty,
                    "confidence_product": math.exp(-penalty),
                    "path_nodes": path_nodes,
                    "canonical_path": canonical_path,
                }
                pair_key = (left, right)
                candidate_rank = (hops, penalty, canonical_path)
                old = best_pair_paths.get(pair_key)
                if old is None:
                    best_pair_paths[pair_key] = candidate
                else:
                    old_rank = (
                        old["hops"],
                        old["penalty"],
                        old["canonical_path"],
                    )
                    if candidate_rank < old_rank:
                        best_pair_paths[pair_key] = candidate

            for neighbour, edge_penalty, _ in adjacency.get(node, ()):  # weak
                new_cost = (hops + 1, penalty + edge_penalty)
                if new_cost < best_cost.get(neighbour, (math.inf, math.inf)):
                    best_cost[neighbour] = new_cost
                    previous[neighbour] = node
                    heapq.heappush(
                        queue,
                        (new_cost[0], new_cost[1], neighbour),
                    )

    return best_pair_paths


def minimum_component_forest(n_components, pair_paths):
    """Kruskal sobre el cierre métrico; devuelve un bosque si el prior es disjunto."""

    parent = list(range(n_components))

    def find(index):
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index

    candidates = sorted(
        pair_paths.values(),
        key=lambda value: (
            value["hops"],
            value["penalty"],
            value["component_a"],
            value["component_b"],
            value["canonical_path"],
        ),
    )
    selected = []
    for candidate in candidates:
        left = candidate["component_a"]
        right = candidate["component_b"]
        root_left, root_right = find(left), find(right)
        if root_left == root_right:
            continue
        parent[root_right] = root_left
        selected.append(candidate)
    return selected


def local_force_layout(nodes, edges, seed=42, iterations=520):
    """Fruchterman-Reingold determinista dentro de una componente."""

    nodes = sorted(nodes)
    n_nodes = len(nodes)
    rng = np.random.default_rng(seed)
    node_index = {node: index for index, node in enumerate(nodes)}

    if n_nodes == 2:
        angle = rng.uniform(0.0, np.pi)
        vector = np.array([np.cos(angle), np.sin(angle)])
        return {nodes[0]: -0.78 * vector, nodes[1]: 0.78 * vector}

    positions = rng.normal(0.0, 0.34, size=(n_nodes, 2))
    optimal_distance = 1.45 / math.sqrt(max(n_nodes, 1))
    initial_temperature = 0.14

    for step in range(iterations):
        delta = positions[:, None, :] - positions[None, :, :]
        distance = np.sqrt((delta**2).sum(axis=2))
        np.fill_diagonal(distance, 1.0)
        safe_distance = np.maximum(distance, 1e-4)
        force = (
            delta
            / safe_distance[:, :, None]
            * (optimal_distance**2 / safe_distance)[:, :, None]
        ).sum(axis=1)

        for source, target in edges:
            source_index = node_index[source]
            target_index = node_index[target]
            vector = positions[source_index] - positions[target_index]
            length = max(float(np.linalg.norm(vector)), 1e-4)
            attraction = vector / length * (length**2 / optimal_distance)
            force[source_index] -= attraction
            force[target_index] += attraction

        norms = np.linalg.norm(force, axis=1)
        temperature = initial_temperature * (1.0 - step / iterations) ** 1.2
        positions += (
            force
            / np.maximum(norms[:, None], 1e-12)
            * np.minimum(norms, temperature)[:, None]
        )
        positions -= positions.mean(axis=0)

    scale = max(float(np.linalg.norm(positions, axis=1).max()), 1e-9)
    positions /= scale
    return {node: positions[node_index[node]] for node in nodes}


def unboxed_forest_layout(nodes, edges, component_list, seed=42):
    """Layout único sin cajas: red principal al centro e islas alrededor."""

    rng = np.random.default_rng(seed)
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))
    component_edges = [
        [edge for edge in edges if edge[0] in component]
        for component in component_list
    ]
    radii = [
        0.025 + 0.030 * math.sqrt(len(component))
        for component in component_list
    ]

    placed = [(np.array([0.0, 0.0]), radii[0])]
    for component_index, radius in enumerate(radii[1:], start=1):
        phase = rng.uniform(-0.15, 0.15)
        for candidate_index in range(1, 20000):
            radial_distance = 0.026 * math.sqrt(candidate_index)
            angle = candidate_index * golden_angle + phase
            candidate = radial_distance * np.array(
                [np.cos(angle), np.sin(angle)]
            )
            if all(
                np.linalg.norm(candidate - center)
                >= radius + other_radius + 0.025
                for center, other_radius in placed
            ):
                placed.append((candidate, radius))
                break
        else:
            raise RuntimeError(
                f"No se pudo colocar la componente {component_index + 1}."
            )

    positions = {}
    for component_index, (component, edges_here) in enumerate(
        zip(component_list, component_edges)
    ):
        center, radius = placed[component_index]
        local = local_force_layout(
            component,
            edges_here,
            seed=seed + 101 * (component_index + 1),
        )
        for node, coordinate in local.items():
            positions[node] = center + coordinate * radius * 0.90

    coordinates = np.vstack([positions[node] for node in nodes])
    coordinates -= coordinates.mean(axis=0)
    scale = max(float(np.linalg.norm(coordinates, axis=1).max()), 1e-9)
    coordinates *= 0.94 / scale
    return {
        node: coordinates[index]
        for index, node in enumerate(nodes)
    }


# ----------------------------------------------------------------------
# 1. Núcleo estricto y componentes originales
# ----------------------------------------------------------------------
strict = strict_edges["Basal-like"].copy()
strict_rows = [
    {
        "source_gene": str(row.source_gene),
        "target_gene": str(row.target_gene),
        "edge_type": int(row.edge_type),
        "prior_weight": float(getattr(row, "prior_weight", np.nan)),
    }
    for row in strict.itertuples(index=False)
]
strict_directed_edges = [
    (row["source_gene"], row["target_gene"], row["edge_type"])
    for row in strict_rows
]
strict_pairs = {
    tuple(sorted((source, target)))
    for source, target, _ in strict_directed_edges
}
strict_nodes = sorted(
    {
        gene
        for source, target, _ in strict_directed_edges
        for gene in (source, target)
    }
)
strict_node_set = set(strict_nodes)
strict_components = weak_components(
    [(source, target) for source, target, _ in strict_directed_edges]
)

strict_contract = (
    len(strict_nodes),
    len(strict_directed_edges),
    len(strict_components),
    len(strict_components[0]),
)
if strict_contract != (288, 299, 75, 49):
    raise RuntimeError(
        "El núcleo no corresponde a V8.1: "
        f"observado={strict_contract}; esperado=(288, 299, 75, 49)."
    )


# ----------------------------------------------------------------------
# 2. Prior completo y caminos mínimos entre componentes
# ----------------------------------------------------------------------
if not Path(SUPP_CONNECT_PRIOR_PATH).is_file():
    raise FileNotFoundError(
        "No se encuentra el prior completo generado por 05: "
        f"{SUPP_CONNECT_PRIOR_PATH}"
    )

prior = pd.read_csv(SUPP_CONNECT_PRIOR_PATH)
pair_meta, prior_adjacency = normalize_prior(prior)
pair_paths = lexicographic_component_paths(
    strict_components,
    prior_adjacency,
)
selected_paths = minimum_component_forest(
    len(strict_components),
    pair_paths,
)


# ----------------------------------------------------------------------
# 3. Unión mínima de aristas y genes puente
# ----------------------------------------------------------------------
bridge_pair_to_paths = defaultdict(list)
path_records = []
for path_id, path in enumerate(selected_paths, start=1):
    path_nodes = list(path["path_nodes"])
    path_pairs = [
        tuple(sorted((left, right)))
        for left, right in zip(path_nodes[:-1], path_nodes[1:])
    ]
    new_pairs = [pair for pair in path_pairs if pair not in strict_pairs]
    intermediate_genes = [
        gene for gene in path_nodes[1:-1] if gene not in strict_node_set
    ]
    for pair in new_pairs:
        bridge_pair_to_paths[pair].append(path_id)
    path_records.append(
        {
            "connector_path_id": path_id,
            "strict_component_a": int(path["component_a"] + 1),
            "strict_component_b": int(path["component_b"] + 1),
            "hops": int(path["hops"]),
            "confidence_product": float(path["confidence_product"]),
            "path_genes": ";".join(path_nodes),
            "intermediate_genes": ";".join(intermediate_genes),
            "new_prior_edges": len(new_pairs),
        }
    )

bridge_records = []
for pair in sorted(bridge_pair_to_paths):
    metadata = pair_meta[pair]
    bridge_records.append(
        {
            "source_gene": metadata["source_gene"],
            "target_gene": metadata["target_gene"],
            "edge_type": metadata["edge_type"],
            "prior_weight": metadata["prior_weight"],
            "sign": metadata["sign"],
            "candidate_edge_id": metadata["candidate_edge_id"],
            "confidence": metadata["confidence"],
            "connector_path_ids": ";".join(
                str(value) for value in bridge_pair_to_paths[pair]
            ),
            "edge_role": "prior_shortest_path_connector",
            "strict_5of5": False,
        }
    )

bridge_edges = pd.DataFrame(bridge_records)
connector_nodes = sorted(
    {
        gene
        for row in bridge_records
        for gene in (row["source_gene"], row["target_gene"])
        if gene not in strict_node_set
    }
)
connector_node_set = set(connector_nodes)

display_pairs = set(strict_pairs) | set(bridge_pair_to_paths)
display_nodes = sorted(
    {gene for edge in display_pairs for gene in edge}
)
display_components = weak_components(sorted(display_pairs))
unreachable_components = display_components[1:]
unreachable_nodes = set().union(*unreachable_components)

# Contrato exacto de esta expansión conservadora V8.1.
completion_contract = {
    "strict_components": len(strict_components),
    "selected_component_paths": len(selected_paths),
    "added_prior_edges": len(bridge_records),
    "added_connector_genes": len(connector_nodes),
    "final_nodes": len(display_nodes),
    "final_weak_components": len(display_components),
    "largest_final_component": len(display_components[0]),
}
expected_completion = {
    "strict_components": 75,
    "selected_component_paths": 71,
    "added_prior_edges": 75,
    "added_connector_genes": 4,
    "final_nodes": 292,
    "final_weak_components": 4,
    "largest_final_component": 286,
}
if completion_contract != expected_completion:
    raise RuntimeError(
        "La expansión no reproduce el contrato V8.1. "
        f"Observado={completion_contract}; esperado={expected_completion}."
    )
if set(connector_nodes) != {"JAK1", "PTEN", "SRC", "STAT3"}:
    raise RuntimeError(
        "Genes puente inesperados: "
        f"{connector_nodes}; esperados JAK1, PTEN, SRC y STAT3."
    )


# ----------------------------------------------------------------------
# 4. Auditoría y exportación tabular
# ----------------------------------------------------------------------
SUPP_CONNECT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(path_records).to_csv(
    SUPP_CONNECT_TABLE_DIR / "basal_prior_connector_paths.csv",
    index=False,
)
bridge_edges.to_csv(
    SUPP_CONNECT_TABLE_DIR / "basal_prior_connector_edges.csv",
    index=False,
)

connector_degree = Counter()
for pair in bridge_pair_to_paths:
    for gene in pair:
        if gene in connector_node_set:
            connector_degree[gene] += 1
pd.DataFrame(
    [
        {
            "gene": gene,
            "connector_degree": connector_degree[gene],
            "node_role": "prior_connector_gene",
        }
        for gene in connector_nodes
    ]
).to_csv(
    SUPP_CONNECT_TABLE_DIR / "basal_prior_connector_genes.csv",
    index=False,
)

unreachable_records = []
for final_rank, component in enumerate(unreachable_components, start=2):
    strict_ranks = [
        index + 1
        for index, strict_component in enumerate(strict_components)
        if strict_component <= component
    ]
    unreachable_records.append(
        {
            "final_component_rank": final_rank,
            "n_genes": len(component),
            "genes": ";".join(sorted(component)),
            "original_strict_component_ranks": ";".join(
                str(value) for value in strict_ranks
            ),
            "reason": "no_path_to_main_component_in_locked_HuRI_OmniPath_prior",
        }
    )
pd.DataFrame(unreachable_records).to_csv(
    SUPP_CONNECT_TABLE_DIR / "basal_prior_unreachable_components.csv",
    index=False,
)

manifest = {
    "analysis": "Basal-like strict 5/5 prior-supported completion",
    "distance": "minimum weak/undirected hop count",
    "tie_break": "maximum product of absolute prior weights",
    "forest": "Kruskal on component metric closure",
    "strict_core": {
        "genes": len(strict_nodes),
        "directed_edges": len(strict_directed_edges),
        "weak_components": len(strict_components),
    },
    "completion": completion_contract,
    "connector_genes": connector_nodes,
    "unreachable_components": [
        sorted(component) for component in unreachable_components
    ],
    "interpretation": (
        "Connector edges and genes come from the locked prior and are not "
        "members of the strict 5/5 learned edge core."
    ),
}
(SUPP_CONNECT_TABLE_DIR / "basal_prior_completion_manifest.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)


# ----------------------------------------------------------------------
# 5. Categorías biológicas y layout final
# ----------------------------------------------------------------------
hallmark_genes = set()
for value in hallmark_raw["overlap_genes"].dropna():
    hallmark_genes.update(
        gene.strip().upper()
        for gene in str(value).split(";")
        if gene.strip()
    )
signature_genes = {
    gene.strip().upper()
    for component_genes in components["basal_signature_genes"].dropna()
    for gene in str(component_genes).split(";")
    if gene.strip()
}


def node_category(node):
    if node in connector_node_set:
        return "connector"
    key = node.upper()
    in_signature = key in signature_genes
    in_hallmark = key in hallmark_genes
    if in_signature and in_hallmark:
        return "signature_and_hallmark"
    if in_signature:
        return "signature_only"
    if in_hallmark:
        return "hallmark_only"
    return "other"


node_categories = {node: node_category(node) for node in display_nodes}
category_counts = Counter(node_categories.values())

# Layout usa pares débiles únicos para evitar duplicar fuerzas por reciprocidad.
layout_edges = sorted(display_pairs)
positions = unboxed_forest_layout(
    display_nodes,
    layout_edges,
    display_components,
    seed=SUPP_CONNECT_LAYOUT_SEED,
)

display_degree = Counter()
for left, right in layout_edges:
    display_degree[left] += 1
    display_degree[right] += 1


# ----------------------------------------------------------------------
# 6. Dibujo
# ----------------------------------------------------------------------
fig_supp_connect, ax = plt.subplots(
    figsize=SUPP_CONNECT_FIGSIZE,
    constrained_layout=True,
)

strict_directed_pair_set = {
    (source, target) for source, target, _ in strict_directed_edges
}
for source, target, edge_type in strict_directed_edges:
    reciprocal = (target, source) in strict_directed_pair_set
    arrow = FancyArrowPatch(
        positions[source],
        positions[target],
        arrowstyle="-|>",
        mutation_scale=3.6 if edge_type == 0 else 4.2,
        linewidth=SUPP_CONNECT_CORE_EDGE_WIDTH,
        color=SUPP_CONNECT_CORE_EDGE_COLORS[edge_type],
        alpha=(
            SUPP_CONNECT_CORE_EDGE_ALPHA
            if edge_type == 0
            else SUPP_CONNECT_CORE_EDGE_ALPHA + 0.12
        ),
        connectionstyle=f"arc3,rad={0.075 if reciprocal else 0.015}",
        shrinkA=2.8,
        shrinkB=2.8,
        zorder=1,
    )
    ax.add_patch(arrow)

# Puentes: dirección real del prior, pero distancia calculada débilmente.
for row in bridge_edges.itertuples(index=False):
    arrow = FancyArrowPatch(
        positions[row.source_gene],
        positions[row.target_gene],
        arrowstyle="-|>",
        mutation_scale=5.3,
        linewidth=SUPP_CONNECT_BRIDGE_EDGE_WIDTH,
        linestyle=(0, (3.0, 2.2)),
        color=SUPP_CONNECT_BRIDGE_EDGE_COLOR,
        alpha=SUPP_CONNECT_BRIDGE_EDGE_ALPHA,
        connectionstyle="arc3,rad=0.022",
        shrinkA=3.0,
        shrinkB=3.0,
        zorder=2,
    )
    ax.add_patch(arrow)

# Nodos por categoría. El borde negro identifica las tres islas sin camino.
for category in (
    "other",
    "hallmark_only",
    "signature_only",
    "signature_and_hallmark",
    "connector",
):
    nodes_here = [
        node for node in display_nodes if node_categories[node] == category
    ]
    if not nodes_here:
        continue
    regular_nodes = [node for node in nodes_here if node not in unreachable_nodes]
    isolated_nodes = [node for node in nodes_here if node in unreachable_nodes]

    for node_group, edge_color, line_width in (
        (regular_nodes, "white", 0.48),
        (isolated_nodes, SUPP_CONNECT_UNREACHABLE_OUTLINE, 1.15),
    ):
        if not node_group:
            continue
        coordinates = np.vstack([positions[node] for node in node_group])
        sizes = [
            (
                SUPP_CONNECT_NODE_BASE_SIZE
                + SUPP_CONNECT_NODE_DEGREE_SCALE * display_degree[node]
            )
            * (
                1.60
                if node in connector_node_set
                else 1.42
                if node.upper() in SUPP_CONNECT_LABELLED_GENES
                else 1.0
            )
            for node in node_group
        ]
        ax.scatter(
            coordinates[:, 0],
            coordinates[:, 1],
            s=sizes,
            c=SUPP_CONNECT_NODE_COLORS[category],
            linewidths=line_width,
            edgecolors=edge_color,
            zorder=4,
        )


genes_to_label = set()
if SUPP_CONNECT_LABEL_ALL_CORE_GENES:
    genes_to_label.update(strict_node_set)
else:
    genes_to_label.update(SUPP_CONNECT_LABELLED_GENES)
if SUPP_CONNECT_LABEL_CONNECTOR_GENES:
    genes_to_label.update(connector_node_set)
if SUPP_CONNECT_LABEL_UNREACHABLE_GENES:
    genes_to_label.update(unreachable_nodes)


def radial_label_offset(node, magnitude):
    vector = np.asarray(positions[node], dtype=float)
    norm = max(float(np.linalg.norm(vector)), 1e-9)
    unit = vector / norm
    return float(unit[0] * magnitude), float(unit[1] * magnitude)


for node in display_nodes:
    if node.upper() not in {gene.upper() for gene in genes_to_label}:
        continue
    selected = node.upper() in SUPP_CONNECT_LABELLED_GENES
    connector = node in connector_node_set
    unreachable = node in unreachable_nodes

    if selected:
        font_size = SUPP_CONNECT_SELECTED_LABEL_SIZE
        font_weight = "bold"
        text_color = "#111111"
        offset = SUPP_CONNECT_LABEL_OFFSETS.get(
            node.upper(), radial_label_offset(node, 22)
        )
    elif connector:
        font_size = SUPP_CONNECT_AUX_LABEL_SIZE
        font_weight = "bold"
        text_color = SUPP_CONNECT_BRIDGE_EDGE_COLOR
        offset = SUPP_CONNECT_LABEL_OFFSETS.get(
            node.upper(), radial_label_offset(node, 19)
        )
    elif unreachable:
        font_size = SUPP_CONNECT_AUX_LABEL_SIZE
        font_weight = "bold"
        text_color = "#111111"
        offset = radial_label_offset(node, 17)
    else:
        font_size = SUPP_CONNECT_ALL_LABEL_SIZE
        font_weight = "normal"
        text_color = "#333333"
        offset = (0, 0)

    label = ax.annotate(
        node,
        xy=positions[node],
        xytext=offset,
        textcoords="offset points",
        fontsize=font_size,
        fontweight=font_weight,
        color=text_color,
        ha="center",
        va="center",
        arrowprops=(
            None
            if offset == (0, 0)
            else {
                "arrowstyle": "-",
                "color": "#777777",
                "linewidth": 0.42,
                "shrinkA": 2,
                "shrinkB": 2,
            }
        ),
        zorder=7,
    )
    label.set_path_effects(
        [
            patheffects.withStroke(
                linewidth=2.0 if font_weight == "bold" else 0.9,
                foreground="white",
            )
        ]
    )


legend_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor=SUPP_CONNECT_NODE_COLORS["signature_and_hallmark"],
        markeredgecolor="white", markersize=6.6,
        label=(
            "Basal signature + Hallmark "
            f"(n={category_counts['signature_and_hallmark']})"
        ),
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor=SUPP_CONNECT_NODE_COLORS["signature_only"],
        markeredgecolor="white", markersize=6.6,
        label=f"Basal signature only (n={category_counts['signature_only']})",
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor=SUPP_CONNECT_NODE_COLORS["hallmark_only"],
        markeredgecolor="white", markersize=6.6,
        label=f"Hallmark only (n={category_counts['hallmark_only']})",
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor=SUPP_CONNECT_NODE_COLORS["other"],
        markeredgecolor="white", markersize=6.6,
        label=f"Neither (n={category_counts['other']})",
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor=SUPP_CONNECT_NODE_COLORS["connector"],
        markeredgecolor="white", markersize=7.2,
        label=f"Prior connector gene (n={len(connector_nodes)})",
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="white",
        markeredgecolor=SUPP_CONNECT_UNREACHABLE_OUTLINE,
        markeredgewidth=1.2, markersize=6.6,
        label=f"Prior-isolated strict gene (n={len(unreachable_nodes)})",
    ),
    Line2D(
        [0], [0], color=SUPP_CONNECT_CORE_EDGE_COLORS[0], lw=1.3,
        label="Strict HuRI",
    ),
    Line2D(
        [0], [0], color=SUPP_CONNECT_CORE_EDGE_COLORS[1], lw=1.3,
        label="Strict OmniPath activation",
    ),
    Line2D(
        [0], [0], color=SUPP_CONNECT_CORE_EDGE_COLORS[2], lw=1.3,
        label="Strict OmniPath inhibition",
    ),
    Line2D(
        [0], [0], color=SUPP_CONNECT_BRIDGE_EDGE_COLOR,
        lw=1.5, linestyle=(0, (3.0, 2.2)),
        label=f"Prior connector edge (n={len(bridge_edges)})",
    ),
]
ax.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.045),
    frameon=False,
    ncol=5,
    fontsize=7.4,
    handletextpad=0.45,
    columnspacing=0.95,
)

ax.set_title(
    SUPP_CONNECT_TITLE + "\n"
    "288 strict-core genes + 4 connector genes · "
    "299 strict edges + 75 prior connector edges",
    fontsize=15.5,
    fontweight="bold",
    pad=14,
)
ax.text(
    0.5,
    -0.087,
    "Dashed blue edges and blue nodes are shortest-path additions from the "
    "locked prior, not members of the learned strict 5/5 core. Black outlines "
    "mark the six strict genes with no path to the main prior component.",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=7.7,
    color="#555555",
)
ax.set_xlim(-1.08, 1.08)
ax.set_ylim(-1.08, 1.08)
ax.set_aspect("equal", adjustable="box")
ax.axis("off")

export_figure(
    fig_supp_connect,
    "Supplementary_Figure_Basal_like_prior_supported_completion",
)
display(fig_supp_connect)

display(pd.DataFrame([completion_contract]))
display(pd.DataFrame(unreachable_records))
display(pd.DataFrame({"connector_gene": connector_nodes}))

print(
    "Expansión conservadora completada:",
    {
        **completion_contract,
        "connector_genes": connector_nodes,
        "unreachable": [
            sorted(component) for component in unreachable_components
        ],
    },
)

Supplementary Figure Sx. Prior-supported completion of the Basal-like strict 5/5 edge core.

The 288 genes and 299 directed edges retained in all five training seeds are shown as the strict core (solid edges). To visualize the relationship among disconnected strict-core components without inventing interactions, components were connected only through paths present in the locked HuRI-OmniPath prior. Paths minimized weak/undirected hop distance; ties were resolved by maximizing the product of absolute prior weights, and a minimum spanning forest was then calculated on the component-level metric closure. This conservative completion added 75 prior-supported connector edges and four intermediary genes (JAK1, PTEN, SRC and STAT3), joining 72 of the 75 original strict components into a main component containing 286 genes. Three strict two-gene components had no path to the main component in the locked prior and therefore remain separate: CD2-CD59, ELF5-GLRX2 and PCID2-SHFM1. Added prior edges are blue and dashed, and intermediary genes are blue; neither is a member of the learned strict 5/5 core. Black node outlines identify the six genes in the three unreachable components. Strict-core node colour indicates membership in the locked directional Basal-like signature and/or in genes contributing to significant Hallmark enrichments (Benjamini-Hochberg FDR < 0.05). Node size is proportional to displayed degree. Layout positions are used only for visualization and do not represent biological distance.

## Figura 3 — Especificidad y coherencia funcional

La matriz usa la firma direccional corregida y el panel inferior los 1,000
rewirings dirigidos que preservan grado, tipo y signo.

La siguiente celda contiene la única implementación activa de la Figura 3.
Se ha retirado la copia duplicada que figuraba en el notebook V7.


In [ ]:
# ===================== EDITAR AQUÍ: FIGURA 3 =====================
FIG3_TITLE = 'Subtype-specific recovery and functional coherence'

FIG3_PANEL_TITLES = [
    'A  Cross-class strict-core specificity',
    'B  Directional Basal-like recovery',
    'C  Functional coherence beyond preserved topology',
]

FIG3_HEATMAP_MIN = 0.0
FIG3_HEATMAP_MAX = 0.28
FIG3_NULL_COLOR = NEUTRAL
FIG3_REWIRING_ERROR_COLOR = NEUTRAL


# ================================================================
# DISEÑO GENERAL
# ================================================================
fig3 = plt.figure(
    figsize=FIGSIZE['biological_validation'],
    constrained_layout=True,
)

grid = fig3.add_gridspec(
    2,
    2,
    height_ratios=[1.0, 0.82],
)


# ================================================================
# PANEL A: ESPECIFICIDAD CRUZADA
# ================================================================
ax_a = fig3.add_subplot(grid[0, 0])

matrix = (
    cross.pivot(
        index='graph_class',
        columns='annotation_class',
        values='node_fraction',
    )
    .loc[CLASSES, CLASSES]
)

image = ax_a.imshow(
    matrix.values,
    cmap=HEATMAP_CMAP,
    vmin=FIG3_HEATMAP_MIN,
    vmax=max(
        FIG3_HEATMAP_MAX,
        float(matrix.values.max()),
    ),
)

for row in range(4):
    for col in range(4):
        value = matrix.iloc[row, col]
        text_color = 'white' if value < 0.15 else 'black'

        ax_a.text(
            col,
            row,
            f'{value:.3f}',
            ha='center',
            va='center',
            color=text_color,
            fontsize=STYLE['annotation_size'],
        )

    ax_a.add_patch(
        Rectangle(
            (row - 0.48, row - 0.48),
            0.96,
            0.96,
            fill=False,
            edgecolor=DIAGONAL_COLOR,
            linewidth=1.8,
        )
    )

ax_a.set_xticks(
    range(4),
    DISPLAY_CLASSES,
    rotation=25,
    ha='right',
)

ax_a.set_yticks(
    range(4),
    DISPLAY_CLASSES,
)

ax_a.set_xlabel('Directional annotation class')
ax_a.set_ylabel('Strict core class')
ax_a.set_title(
    FIG3_PANEL_TITLES[0],
    loc='left',
)

fig3.colorbar(
    image,
    ax=ax_a,
    fraction=0.046,
    pad=0.04,
)


# ================================================================
# PANEL B: RECUPERACIÓN DIRECCIONAL BASAL-LIKE
# ================================================================
ax_b = fig3.add_subplot(grid[0, 1])

basal = matched.loc[
    matched['graph_class'].eq('Basal-like')
    & matched['term'].eq('Basal-like')
].copy()

statistic_order = [
    'at_least_one_endpoint',
    'both_endpoints',
    'unique_node_fraction',
]

statistic_labels = [
    '≥1 endpoint',
    'Both endpoints',
    'Unique nodes',
]

basal = (
    basal
    .set_index('statistic')
    .loc[statistic_order]
    .reset_index()
)

x_b = np.arange(3)
width_b = 0.36

ax_b.bar(
    x_b - width_b / 2,
    basal['observed'],
    width_b,
    color=CLASS_COLORS['Basal-like'],
    label='Observed',
)

ax_b.bar(
    x_b + width_b / 2,
    basal['null_mean'],
    width_b,
    color=FIG3_NULL_COLOR,
    label='Matched-null mean',
)

for index, row in basal.iterrows():
    ax_b.text(
        index,
        max(row.observed, row.null_mean) + 0.018,
        (
            f'{row.fold_over_null:.2f}×\n'
            f'FDR={row.fdr_global:.3f}'
        ),
        ha='center',
        va='bottom',
        fontsize=STYLE['annotation_size'],
    )

ax_b.set_xticks(
    x_b,
    statistic_labels,
)

ax_b.set_ylim(
    0,
    max(
        0.54,
        float(
            basal[
                ['observed', 'null_mean']
            ].to_numpy().max()
        ) * 1.28,
    ),
)

ax_b.set_ylabel('Fraction')
ax_b.set_title(
    FIG3_PANEL_TITLES[1],
    loc='left',
)

ax_b.legend(
    frameon=False,
    loc='upper right',
)

ax_b.spines[['top', 'right']].set_visible(False)


# ================================================================
# PANEL C: COHERENCIA FUNCIONAL FRENTE A REWIRINGS
# ================================================================
ax_c = fig3.add_subplot(grid[1, :])

rewiring = (
    rewired
    .set_index('graph_class')
    .loc[CLASSES]
)

x_c = np.arange(4)
width_c = 0.36

# Las barras observadas llevan el color de cada subtipo.
# No se incluyen como una única entrada azul en la leyenda.
ax_c.bar(
    x_c - width_c / 2,
    rewiring['observed_coherent_fraction'],
    width_c,
    color=[
        CLASS_COLORS[class_name]
        for class_name in CLASSES
    ],
    label='Observed',
)

# Distribución nula obtenida mediante 1,000 rewirings.
ax_c.bar(
    x_c + width_c / 2,
    rewiring['null_mean'],
    width_c,
    yerr=rewiring['null_sd'],
    color=FIG3_REWIRING_ERROR_COLOR,
    ecolor='black',
    capsize=3,
    error_kw={
        'elinewidth': 1.2,
        'capthick': 1.2,
    },
    label='Rewiring null (mean ± SD; n=1,000)',
)

# Altura máxima de cada comparación, incluyendo la SD del nulo.
annotation_tops = np.maximum(
    rewiring[
        'observed_coherent_fraction'
    ].to_numpy(dtype=float),
    (
        rewiring['null_mean']
        + rewiring['null_sd']
    ).to_numpy(dtype=float),
)

# Margen superior para evitar que 2.08× se solape con el título.
ax_c.set_ylim(
    0,
    float(annotation_tops.max()) * 1.22,
)

for index, row in enumerate(rewiring.itertuples()):
    annotation_y = max(
        row.observed_coherent_fraction,
        row.null_mean + row.null_sd,
    )

    ax_c.annotate(
        f'{row.fold_over_null:.2f}×',
        xy=(index, annotation_y),
        xytext=(0, 6),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=STYLE['annotation_size'],
    )

ax_c.set_xticks(
    x_c,
    DISPLAY_CLASSES,
)

ax_c.set_ylabel(
    'Hallmark-coherent edge fraction'
)

ax_c.set_title(
    FIG3_PANEL_TITLES[2],
    loc='left',
)

ax_c.legend(
    frameon=False,
    loc='upper right',
)

ax_c.spines[['top', 'right']].set_visible(False)


# ================================================================
# TÍTULO GENERAL
# ================================================================
fig3.suptitle(
    FIG3_TITLE,
    fontsize=STYLE['figure_title_size'],
    fontweight='bold',
)


# ================================================================
# ALINEACIÓN FINAL
# ================================================================
# constrained_layout debe calcular primero las posiciones reales,
# incluyendo el aspecto cuadrado del heatmap y su barra de color.
fig3.canvas.draw()

position_a = ax_a.get_position()
position_b = ax_b.get_position()
position_c = ax_c.get_position()

# Congela el diseño para que savefig no recalcule las posiciones.
fig3.set_layout_engine('none')

# El eje vertical de C coincide exactamente con el borde izquierdo
# del heatmap A. Su extremo derecho coincide con el panel B.
ax_c.set_position([
    position_a.x0,
    position_c.y0,
    position_b.x1 - position_a.x0,
    position_c.height,
])


# ================================================================
# EXPORTACIÓN
# ================================================================
export_figure(
    fig3,
    'Figure_biological_validation',
)

display(fig3)

## Figura 4 — Concordancia estructural externa

Utiliza el núcleo V8.1 Basal-like de 299 aristas, fijado antes de la
estimación externa. Se muestran las correlaciones observadas y los nulos
emparejados por grado dirigido, tipo de arista y signo del prior.


In [ ]:
# ===================== EDITAR AQUÍ: FIGURA 4 =====================
FIG4_TITLE = 'External concordance of Basal-like edge effects'
FIG4_XLABEL = 'Spearman correlation: development vs external edge effects'
FIG4_SCOPE_ORDER = ['TCGA-BRCA', 'GSE21653', 'GSE1456', 'EXTERNAL_META']
FIG4_SCOPE_LABELS = [
    'TCGA-BRCA',
    'GSE21653',
    'GSE1456',
    'External meta-analysis',
]
FIG4_XLIM = (0.05, 0.76)
FIG4_INTERVAL_COLOR = '#BDBDBD'
FIG4_NULL_COLOR = '#4D4D4D'
FIG4_CORE_COLOR = CLASS_COLORS['Basal-like']
FIG4_LEGEND_ANCHOR = (0.03, 0.56)

external = external_raw.copy()
primary = external.loc[external['is_primary_metric'].astype(bool)].copy()
primary = (
    primary.set_index('scope').loc[FIG4_SCOPE_ORDER].reset_index()
)
y_positions = np.arange(len(primary))[::-1]

fig4, ax = plt.subplots(
    figsize=FIGSIZE['external_concordance'],
    constrained_layout=True,
)
ax.hlines(
    y_positions,
    primary['null_q025'],
    primary['null_q975'],
    color=FIG4_INTERVAL_COLOR,
    lw=7,
    label='Matched-null 95% interval',
)
ax.scatter(
    primary['null_mean'],
    y_positions,
    color=FIG4_NULL_COLOR,
    s=48,
    zorder=3,
    label='Matched-null mean',
)
ax.scatter(
    primary['observed'],
    y_positions,
    color=FIG4_CORE_COLOR,
    s=82,
    zorder=4,
    label='Basal-like 5/5 core',
)
for y, row in zip(y_positions, primary.itertuples(index=False)):
    ax.text(
        min(0.72, row.observed + 0.016),
        y,
        f'FDR={row.fdr_primary_four_scopes:.3f}',
        va='center',
        ha='left',
        fontsize=STYLE['annotation_size'],
    )
ax.set_yticks(y_positions, FIG4_SCOPE_LABELS)
ax.set_xlim(*FIG4_XLIM)
ax.set_xlabel(FIG4_XLABEL)
ax.set_title(FIG4_TITLE, loc='left', fontweight='bold')
ax.legend(
    frameon=False,
    loc='center left',
    bbox_to_anchor=FIG4_LEGEND_ANCHOR,
)
ax.spines[['top', 'right']].set_visible(False)

export_figure(fig4, 'Figure_external_structural_concordance')
display(fig4)

## Figura 5 — Ablaciones, factorial con q y sensibilidad estructural

Los seis paneles muestran las cuatro ablaciones atómicas frente a `FULL`,
el factorial aprendizaje × heurística modular, la curva de intersección
`k/R` y la sensibilidad a todos los subconjuntos de semillas. Ninguno de
estos paneles utiliza las permutaciones de etiquetas.


In [ ]:
# ===================== EDITAR AQUÍ: FIGURA 5 =====================
FIG5_TITLE = 'Mechanism controls and parameter sensitivity'
FIG5_PANEL_TITLES = [
    'A  Five-seed reproducibility',
    'B  Strict 5/5 core size',
    'C  Directional Basal-like recovery',
    'D  Learning × module factorial',
    'E  Core threshold curve',
    'F  Seed-subset sensitivity',
]

ARM_ORDER = [
    'FULL',
    'NO_CONTEXT_LOSS',
    'NO_COHORT_QUERY_LOSS',
    'NO_EDGE_TYPE_CALIBRATION',
    'NO_SIGNED_WEIGHT',
]
# Etiquetas metodológicamente exactas. La última ablación conserva las
# etiquetas de relación y elimina únicamente el signo del peso escalar prior.
ARM_LABELS = [
    'Full',
    'No context\nloss',
    'No cohort-query\nloss',
    'No edge-type\ncalibration',
    'No signed scalar\nprior weight',
]
ARM_COLORS = ['#2166AC', '#B2182B', '#969696', '#969696', '#969696']
REFERENCE_LINE_COLOR = '#4D4D4D'

ablation = ablation_raw.set_index('arm').loc[ARM_ORDER]
unique_tests = (
    ablation_tests.loc[
        ablation_tests['statistic'].eq('unique_node_fraction')
    ].set_index('arm').loc[ARM_ORDER]
)
fdr_column = 'fdr_global_15_tests'

fig5, axes = plt.subplots(
    2, 3,
    figsize=FIGSIZE['controls_sensitivity'],
    constrained_layout=True,
)
x = np.arange(len(ARM_ORDER))

# Panel A: reproducibilidad de las ablaciones.
ax = axes[0, 0]
ax.bar(x, ablation['mean_pairwise_edge_jaccard'], color=ARM_COLORS)
ax.set_xticks(x, ARM_LABELS, rotation=22, ha='right')
ax.set_ylim(
    0,
    max(0.1, float(ablation['mean_pairwise_edge_jaccard'].max()) * 1.18),
)
ax.set_ylabel('Mean pairwise edge Jaccard')
ax.set_title(FIG5_PANEL_TITLES[0], loc='left')
ax.spines[['top', 'right']].set_visible(False)

# Panel B: tamaño absoluto del núcleo estricto.
ax = axes[0, 1]
bars = ax.bar(x, ablation['strict_edges'], color=ARM_COLORS)
offset = max(1.0, float(ablation['strict_edges'].max()) * 0.025)
for bar, value in zip(bars, ablation['strict_edges']):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + offset,
        f'{int(value)}',
        ha='center',
        fontsize=STYLE['annotation_size'],
    )
ax.set_xticks(x, ARM_LABELS, rotation=22, ha='right')
ax.set_ylim(0, max(1.0, float(ablation['strict_edges'].max()) * 1.18))
ax.set_ylabel('Directed edges')
ax.set_title(FIG5_PANEL_TITLES[1], loc='left')
ax.spines[['top', 'right']].set_visible(False)

# Panel C: recuperación direccional frente al nulo candidato emparejado.
ax = axes[0, 2]
bars = ax.bar(x, unique_tests['fold_over_null'], color=ARM_COLORS)
for bar, row in zip(bars, unique_tests.itertuples()):
    fdr = float(getattr(row, fdr_column))
    annotation = 'ns' if fdr >= 0.05 else f'FDR={fdr:.3f}'
    ax.text(
        bar.get_x() + bar.get_width()/2,
        row.fold_over_null + 0.07,
        annotation,
        ha='center',
        fontsize=7.8,
    )
ax.axhline(1.0, color=REFERENCE_LINE_COLOR, lw=0.9, ls='--')
ax.set_xticks(x, ARM_LABELS, rotation=22, ha='right')
ax.set_ylim(0, max(1.2, float(unique_tests['fold_over_null'].max()) * 1.22))
ax.set_ylabel('Unique-node fold over matched null')
ax.set_title(FIG5_PANEL_TITLES[2], loc='left')
ax.spines[['top', 'right']].set_visible(False)

# Panel D: factorial aprendizaje × heurística modular; no usa permutaciones.
statistics = [
    'at_least_one_endpoint',
    'both_endpoints',
    'unique_node_fraction',
]
stat_labels = ['>=1 endpoint', 'Both endpoints', 'Unique nodes']
factorial_order = ['InterGATE_raw', 'InterGATE_module', 'q_raw', 'q_module']
factorial_labels = ['InterGATE raw', 'InterGATE module', 'q raw', 'q module']
factorial_colors = ['#2F6FB0', '#103D6E', '#E58B2A', '#A94F00']
ax = axes[1, 0]
stat_x = np.arange(3)
width = 0.19
offsets = np.linspace(-1.5 * width, 1.5 * width, len(factorial_order))
for arm, label, color, bar_offset in zip(
    factorial_order, factorial_labels, factorial_colors, offsets
):
    values = (
        factorial.loc[factorial['arm'].eq(arm)]
        .set_index('statistic').loc[statistics, 'fold_over_null']
    )
    ax.bar(stat_x + bar_offset, values, width, color=color, label=label)
ax.axhline(1.0, color=REFERENCE_LINE_COLOR, lw=0.9, ls='--')
ax.set_xticks(stat_x, stat_labels, rotation=15, ha='right')
ax.set_ylabel('Fold over matched null')
ax.set_title(FIG5_PANEL_TITLES[3], loc='left')
ax.legend(frameon=False, fontsize=7.2, ncol=2)
ax.spines[['top', 'right']].set_visible(False)

# Panel E: curva k/R para selección raw y modular.
ax = axes[1, 1]
for selector, color in (('raw', '#4C78A8'), ('module', '#C44E52')):
    block = threshold.loc[threshold['selector'].eq(selector)].sort_values('k')
    ax.plot(
        block['k'], block['n_edges'], marker='o', lw=2,
        color=color, label=selector,
    )
r_value = int(threshold['R'].iloc[0])
ax.set_xticks(range(1, r_value + 1))
ax.set_xlabel(f'Minimum selections k (R={r_value})')
ax.set_ylabel('Edges retained at >=k/R')
ax.set_title(FIG5_PANEL_TITLES[4], loc='left')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)

# Panel F: distribución de intersecciones R/R en todos los subconjuntos.
ax = axes[1, 2]
positions = np.arange(2, r_value + 1)
for bar_offset, selector, color in (
    (-0.13, 'raw', '#4C78A8'),
    (0.13, 'module', '#C44E52'),
):
    blocks = [
        subsets.loc[
            subsets['selector'].eq(selector)
            & subsets['subset_R'].eq(r),
            'strict_edges_RofR',
        ].to_numpy(float)
        for r in positions
    ]
    box = ax.boxplot(
        blocks,
        positions=positions + bar_offset,
        widths=0.22,
        patch_artist=True,
        manage_ticks=False,
        showfliers=False,
    )
    for patch in box['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.65)
    for median in box['medians']:
        median.set_color('white')
ax.plot([], [], color='#4C78A8', lw=7, alpha=0.65, label='raw')
ax.plot([], [], color='#C44E52', lw=7, alpha=0.65, label='module')
ax.set_xticks(positions)
ax.set_xlabel('Number of seeds in subset R')
ax.set_ylabel('Strict R/R intersection size')
ax.set_title(FIG5_PANEL_TITLES[5], loc='left')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)

fig5.suptitle(
    FIG5_TITLE,
    fontsize=STYLE['figure_title_size'],
    fontweight='bold',
)
export_figure(fig5, 'Figure_controls_sensitivity')
display(fig5)


## Comprobación final

Verifica los formatos PDF, SVG y PNG a 600 dpi. Las figuras utilizan V8.1
y no requieren el resumen de las permutaciones de etiquetas.


In [ ]:
figure_specs = [
    ('Figure_structural_recovery', 'fig1'),
    ('Figure_Basal_strict_core', 'fig2'),
    ('Figure_biological_validation', 'fig3'),
    ('Figure_external_structural_concordance', 'fig4'),
    ('Figure_controls_sensitivity', 'fig5'),
]

rows = []
for stem, object_name in figure_specs:
    formats = {
        extension: (FIGURES / f'{stem}.{extension}').is_file()
        for extension in ('pdf', 'svg', 'png')
    }
    rows.append({
        'figure': stem,
        'matplotlib_object': object_name,
        **formats,
        'status': 'complete' if all(formats.values()) else 'missing',
    })

status = pd.DataFrame(rows)
display(status)
if not status[['pdf', 'svg', 'png']].to_numpy().all():
    raise RuntimeError('Falta al menos un formato de alguna figura.')
print('Las cinco figuras están completas y siguen abiertas para edición.')